In [1]:
%pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
import hashlib
import json
import torch

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def read_json(path):
    return json.loads(path.read_text())

INPUT_ROOT = Path("/kaggle/input")

baseline_matches = sorted({
    p.parent
    for p in INPUT_ROOT.rglob("initialization_manifest.json")
    if p.parent.name == "HealthVer_TinyLlama_Experiments"
})
assert len(baseline_matches) == 1, (
    f"Expected one attached TinyLlama baseline; found: {baseline_matches}"
)
TINY_BASELINE_ROOT = baseline_matches[0]
BASELINE_RUN = TINY_BASELINE_ROOT / "TinyLlama_LoRA_full_seed42"

required = [
    "input_config.json",
    "preprocessing_manifest.json",
    "dataset_metadata.json",
    "tokenizer/tokenizer_config.json",
    "tokenized_train/state.json",
    "tokenized_validation/state.json",
    "initial_adapter_seed42/adapter_config.json",
    "initial_adapter_seed42/adapter_model.safetensors",
    "TinyLlama_LoRA_full_seed42/run_config.json",
    "TinyLlama_LoRA_full_seed42/training_complete.json",
    "TinyLlama_LoRA_full_seed42/best_adapter/adapter_model.safetensors",
    "full_data_results.csv",
]
for relative in required:
    assert (TINY_BASELINE_ROOT / relative).is_file(), relative

input_config = read_json(TINY_BASELINE_ROOT / "input_config.json")
preprocessing = read_json(
    TINY_BASELINE_ROOT / "preprocessing_manifest.json"
)
initialization = read_json(
    TINY_BASELINE_ROOT / "initialization_manifest.json"
)
baseline_config = read_json(BASELINE_RUN / "run_config.json")
completion = read_json(BASELINE_RUN / "training_complete.json")

assert input_config["model"] == "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
assert input_config["model_revision"] == (
    "fe8a4ea1ffedaf415f4da2f062534de366a451e6"
)
assert input_config["max_sequence_length"] == 448
assert input_config["truncation"] is False
assert initialization["optimizer_updates"] == 0

assert sha256(
    TINY_BASELINE_ROOT / "initial_adapter_seed42/adapter_model.safetensors"
) == initialization["initial_adapter_sha256"] == (
    baseline_config["initial_adapter_sha256"]
)
assert sha256(
    BASELINE_RUN / "best_adapter/adapter_model.safetensors"
) == completion["best_adapter_sha256"]

assert sha256(
    TINY_BASELINE_ROOT / "input_config.json"
) == baseline_config["input_config_sha256"]
assert sha256(
    TINY_BASELINE_ROOT / "preprocessing_manifest.json"
) == baseline_config["preprocessing_manifest_sha256"]

revision = input_config["dataset_revision"]
for split, filename, expected_rows in [
    ("train", "healthver_train.csv", 10590),
    ("validation", "healthver_dev.csv", 1917),
]:
    csv_path = (
        TINY_BASELINE_ROOT / "original_data"
        / f"{revision}_{filename}"
    )
    saved = preprocessing["splits"][split]
    assert saved["rows"] == expected_rows
    assert sha256(csv_path) == saved["source_csv_sha256"]
    print(f"{split}: {expected_rows:,} rows recorded; CSV hash verified")

SUBSET_SIZES = {
    "50pct": 5295, "25pct": 2648, "10pct": 1059, "5pct": 530
}
random_matches = sorted({
    p.parent
    for p in INPUT_ROOT.rglob("subset_50pct.json")
    if p.parent.name == "stratified_random_seed42"
    and p.parent.parent.name == "HealthVer_Random_Experiments"
})
assert len(random_matches) == 1, (
    f"Expected one original random-selection directory: {random_matches}"
)
ORIGINAL_SELECTION_DIR = random_matches[0]

for name in SUBSET_SIZES:
    assert (ORIGINAL_SELECTION_DIR / f"subset_{name}.json").is_file()

print("\nTinyLlama baseline:", TINY_BASELINE_ROOT)
print("Original random selections:", ORIGINAL_SELECTION_DIR)
print("Initial and best adapter hashes: PASSED")
print("Sequence limit: 448; no truncation")

print("\nEnvironment:")
for package in [
    "torch", "transformers", "peft", "accelerate",
    "datasets", "scikit-learn", "torchao",
]:
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

assert torch.cuda.is_available(), "Enable the Kaggle GPU accelerator."
for index in range(torch.cuda.device_count()):
    print(f"GPU {index}: {torch.cuda.get_device_name(index)}")

print("\nCheck complete. No model loaded or files written.")
print("Test contents not accessed. No new selection or training performed.")

train: 10,590 rows recorded; CSV hash verified
validation: 1,917 rows recorded; CSV hash verified

TinyLlama baseline: /kaggle/input/datasets/datasciencexlscout/base-3/HealthVer_TinyLlama_Experiments
Original random selections: /kaggle/input/datasets/datasciencexlscout/random-1/HealthVer_Random_Experiments/stratified_random_seed42
Initial and best adapter hashes: PASSED
Sequence limit: 448; no truncation

Environment:
torch: 2.11.0+cu128
transformers: 5.16.1
peft: 0.20.0
accelerate: 1.14.0
datasets: 4.8.5
scikit-learn: 1.6.1
torchao: NOT INSTALLED
GPU 0: Tesla T4
GPU 1: Tesla T4

Check complete. No model loaded or files written.
Test contents not accessed. No new selection or training performed.


In [3]:
import pandas as pd
from datasets import load_from_disk

OUTPUT_DIR = Path("/kaggle/working/HealthVer_TinyLlama_Random_Experiments")
SELECTION_DIR = OUTPUT_DIR / "stratified_random_seed42"
LABELS = ["Supports", "Refutes", "Neutral"]
DATASET_REVISION = input_config["dataset_revision"]

def content_hash(dataset):
    digest = hashlib.sha256()
    for row in dataset:
        encoded = json.dumps(
            row, sort_keys=True, separators=(",", ":")
        ) + "\n"
        digest.update(encoded.encode("utf-8"))
    return digest.hexdigest()

full_train = load_from_disk(
    str(TINY_BASELINE_ROOT / "tokenized_train")
)
full_validation = load_from_disk(
    str(TINY_BASELINE_ROOT / "tokenized_validation")
)

raw_frames = {}
verified_hashes = {}

for split, dataset, filename, expected_rows in [
    ("train", full_train, "healthver_train.csv", 10590),
    ("validation", full_validation, "healthver_dev.csv", 1917),
]:
    source = (
        TINY_BASELINE_ROOT / "original_data"
        / f"{DATASET_REVISION}_{filename}"
    )
    saved = preprocessing["splits"][split]
    assert sha256(source) == saved["source_csv_sha256"]

    frame = pd.read_csv(source, dtype=str, keep_default_na=False)
    assert len(frame) == len(dataset) == expected_rows
    assert frame["id"].is_unique
    assert set(frame["label"]).issubset(LABELS)
    assert list(dataset["id"]) == frame["id"].tolist()

    # Verify actual contents, not the reload-dependent cache fingerprint.
    actual_hash = content_hash(dataset)
    assert actual_hash == saved["tokenized_content_sha256"], split
    assert max(dataset["sequence_length"]) <= 448

    state = read_json(
        TINY_BASELINE_ROOT / f"tokenized_{split}" / "state.json"
    )
    assert state["_fingerprint"] == saved["dataset_fingerprint"]

    raw_frames[split] = frame
    verified_hashes[split] = actual_hash
    print(f"{split}: all {len(dataset):,} tokenized rows verified")

assert set(raw_frames["train"]["id"]).isdisjoint(
    raw_frames["validation"]["id"]
)

TRAIN_SUBSETS = {}
selection_hashes = {}
pending_copies = []
summary = []

for name, size in SUBSET_SIZES.items():
    source = ORIGINAL_SELECTION_DIR / f"subset_{name}.json"
    manifest = read_json(source)

    assert manifest["method"] == "stratified_random"
    assert manifest["selection_seed"] == 42
    assert manifest["dataset_revision"] == DATASET_REVISION
    assert manifest["source_split"] == "train"
    assert manifest["actual_rows"] == size

    indices = manifest["row_indices"]
    assert len(indices) == len(set(indices)) == size
    assert all(type(i) is int and 0 <= i < len(full_train)
               for i in indices)
    assert indices == sorted(indices)

    selected = full_train.select(indices)
    selected_frame = raw_frames["train"].iloc[indices]
    assert list(selected["id"]) == manifest["ids"]
    assert manifest["ids"] == selected_frame["id"].tolist()

    destination = SELECTION_DIR / source.name
    source_hash = sha256(source)
    if destination.exists():
        assert sha256(destination) == source_hash, (
            f"Existing selection differs: {destination}"
        )

    TRAIN_SUBSETS[name] = selected
    selection_hashes[name] = source_hash
    pending_copies.append((source, destination))

    counts = selected_frame["label"].value_counts()
    summary.append({
        "subset": name,
        "rows": len(selected),
        **{label: int(counts.get(label, 0)) for label in LABELS},
    })

provenance = {
    "model": input_config["model"],
    "model_revision": input_config["model_revision"],
    "dataset_revision": DATASET_REVISION,
    "baseline_root": str(TINY_BASELINE_ROOT),
    "original_selection_directory": str(ORIGINAL_SELECTION_DIR),
    "sequence_limit": 448,
    "truncation": False,
    "selection_seed": 42,
    "verified_content_sha256": verified_hashes,
    "selection_manifest_sha256": selection_hashes,
    "initial_adapter_sha256": initialization["initial_adapter_sha256"],
}

provenance_path = OUTPUT_DIR / "selection_provenance.json"
if provenance_path.exists():
    assert read_json(provenance_path) == provenance, (
        "Existing provenance differs; inspect before continuing."
    )

# Write only after every subset passes verification.
SELECTION_DIR.mkdir(parents=True, exist_ok=True)
for source, destination in pending_copies:
    if not destination.exists():
        destination.write_bytes(source.read_bytes())
    assert sha256(destination) == sha256(source)

if not provenance_path.exists():
    provenance_path.write_text(
        json.dumps(provenance, indent=2), encoding="utf-8"
    )

print("\n" + pd.DataFrame(summary).to_string(index=False))
print("\nVerified: identical Qwen selection indices, IDs and manifest hashes.")
print("Shared validation:", len(full_validation))
print("Saved:", SELECTION_DIR)
print("No model weights loaded, training or test evaluation performed.")

train: all 10,590 tokenized rows verified
validation: all 1,917 tokenized rows verified

subset  rows  Supports  Refutes  Neutral
 50pct  5295      1891     1206     2198
 25pct  2648       946      603     1099
 10pct  1059       378      241      440
  5pct   530       189      121      220

Verified: identical Qwen selection indices, IDs and manifest hashes.
Shared validation: 1917
Saved: /kaggle/working/HealthVer_TinyLlama_Random_Experiments/stratified_random_seed42
No model weights loaded, training or test evaluation performed.


In [4]:
# Append this cell after the verified TinyLlama random-subset preparation cell.
import gc
import json
import math
import time
import hashlib
from pathlib import Path
from importlib.metadata import version

import pandas as pd
import torch
from torch.nn.utils.rnn import pad_sequence
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, GenerationConfig,
    Trainer, TrainingArguments, set_seed,
)
from transformers.trainer_utils import get_last_checkpoint
from peft import PeftModel
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix,
)
from tqdm.auto import tqdm

assert all(name in globals() for name in [
    'TINY_BASELINE_ROOT', 'OUTPUT_DIR', 'SELECTION_DIR',
    'full_train', 'full_validation',
]), 'Run the preflight and corrected subset-preparation cells first.'
DEVICE = torch.device('cuda:0')
assert torch.cuda.is_available()
torch.cuda.set_device(DEVICE)

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(8 * 1024**2), b''):
            h.update(chunk)
    return h.hexdigest()

def json_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()

def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False), encoding='utf-8')
    temporary.replace(path)

def content_hash(dataset):
    h = hashlib.sha256()
    for row in dataset:
        h.update((json.dumps(row, sort_keys=True, separators=(',', ':')) + '\n').encode('utf-8'))
    return h.hexdigest()

input_config = json.loads((TINY_BASELINE_ROOT / 'input_config.json').read_text())
baseline_config = json.loads((TINY_BASELINE_ROOT / 'TinyLlama_LoRA_full_seed42/run_config.json').read_text())
baseline_completion = json.loads((TINY_BASELINE_ROOT / 'TinyLlama_LoRA_full_seed42/training_complete.json').read_text())
dataset_metadata = json.loads((TINY_BASELINE_ROOT / 'dataset_metadata.json').read_text())
provenance = json.loads((OUTPUT_DIR / 'selection_provenance.json').read_text())
preprocessing = json.loads((TINY_BASELINE_ROOT / 'preprocessing_manifest.json').read_text())
assert sha256(TINY_BASELINE_ROOT / 'preprocessing_manifest.json') == baseline_config['preprocessing_manifest_sha256']
assert json_hash(baseline_config) == baseline_completion['run_config_sha256']
assert provenance['model'] == input_config['model']
assert provenance['model_revision'] == input_config['model_revision']
assert provenance['dataset_revision'] == input_config['dataset_revision']
assert provenance['sequence_limit'] == 448 and provenance['truncation'] is False
for split in ['train', 'validation']:
    assert provenance['verified_content_sha256'][split] == preprocessing['splits'][split]['tokenized_content_sha256']
assert sha256(TINY_BASELINE_ROOT / 'input_config.json') == baseline_config['input_config_sha256']

MODEL_ID = input_config['model']
MODEL_REVISION = input_config['model_revision']
DATASET_REVISION = input_config['dataset_revision']
LABELS = input_config['labels']
SEQUENCE_LIMIT = input_config['max_sequence_length']
assert MODEL_ID == 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
assert MODEL_REVISION == 'fe8a4ea1ffedaf415f4da2f062534de366a451e6'
assert DATASET_REVISION == dataset_metadata['revision'] == 'b20ac99ceed62f5264a319fa25a854df1668d85b'
assert LABELS == ['Supports', 'Refutes', 'Neutral']
assert SEQUENCE_LIMIT == 448 and input_config['truncation'] is False
assert input_config['loss'] == 'assistant_answer_only'
assert (TINY_BASELINE_ROOT / 'original_data' / f'{DATASET_REVISION}_healthver_test.csv').is_file(), 'Saved baseline is missing the original test CSV.'

SEED = baseline_config['seed']
EPOCHS = baseline_config['epochs']
MICRO_BATCH = baseline_config['micro_batch_size']
GRAD_ACCUM = baseline_config['gradient_accumulation']
assert (SEED, EPOCHS, MICRO_BATCH, GRAD_ACCUM) == (42, 3, 8, 4)
assert baseline_config['learning_rate'] == 1e-4
for key, value in [('optimizer', 'adamw_torch'), ('weight_decay', 0.01),
                   ('max_grad_norm', 1.0), ('scheduler', 'linear')]:
    assert baseline_config[key] == value

package_versions = {name: version(name) for name in [
    'torch', 'transformers', 'peft', 'accelerate', 'datasets', 'scikit-learn',
]}
assert package_versions == {
    'torch': '2.11.0+cu128', 'transformers': '5.16.1', 'peft': '0.20.0',
    'accelerate': '1.14.0', 'datasets': '4.8.5', 'scikit-learn': '1.6.1',
}, package_versions

INITIAL_ADAPTER = TINY_BASELINE_ROOT / 'initial_adapter_seed42'
initialization = json.loads((TINY_BASELINE_ROOT / 'initialization_manifest.json').read_text())
initial_hashes = {name: sha256(INITIAL_ADAPTER / name) for name in [
    'adapter_config.json', 'adapter_model.safetensors',
]}
assert initial_hashes['adapter_model.safetensors'] == initialization['initial_adapter_sha256']
assert initial_hashes['adapter_model.safetensors'] == baseline_config['initial_adapter_sha256']
assert initial_hashes['adapter_config.json'] == baseline_config['initial_adapter_config_sha256']
assert initial_hashes['adapter_model.safetensors'] == provenance['initial_adapter_sha256']
assert initialization['optimizer_updates'] == 0
assert (baseline_config['lora_rank'], baseline_config['lora_alpha'], baseline_config['lora_dropout']) == (16, 32, 0.05)
assert baseline_config['target_modules'] == 'all-linear'
assert baseline_config['attention_implementation'] == 'sdpa'
assert baseline_config['gradient_checkpointing'] is False
assert baseline_config['evaluation_batch_size'] == 8
assert baseline_config['selection_metric'] == 'validation_macro_f1'
assert initialization['trainable_parameters'] == 12615680

tokenizer = AutoTokenizer.from_pretrained(str(TINY_BASELINE_ROOT / 'tokenizer'), local_files_only=True)
assert tokenizer.pad_token_id is not None
assert hashlib.sha256(tokenizer.chat_template.encode()).hexdigest() == baseline_config['chat_template_sha256']
generation_config = GenerationConfig.from_dict(baseline_config['generation_config'])
assert generation_config.max_new_tokens == 12
assert generation_config.do_sample is False and generation_config.num_beams == 1
assert generation_config.pad_token_id == tokenizer.pad_token_id
assert generation_config.eos_token_id == tokenizer.eos_token_id

def load_original(split, filename):
    path = TINY_BASELINE_ROOT / 'original_data' / f'{DATASET_REVISION}_{filename}'
    assert sha256(path) == dataset_metadata['files'][split]['sha256']
    frame = pd.read_csv(path, dtype=str, keep_default_na=False)
    assert frame['id'].is_unique and set(frame['label']).issubset(LABELS)
    return frame

train_frame = load_original('train', 'healthver_train.csv')
validation_frame = load_original('validation', 'healthver_dev.csv')
assert len(train_frame) == len(full_train) == 10590
assert len(validation_frame) == len(full_validation) == 1917
assert list(full_train['id']) == train_frame['id'].tolist()
assert list(full_validation['id']) == validation_frame['id'].tolist()
assert set(train_frame['id']).isdisjoint(validation_frame['id'])
# Stable content checks replace the incorrect loaded-cache-fingerprint check.
assert content_hash(full_train) == provenance['verified_content_sha256']['train']
assert content_hash(full_validation) == provenance['verified_content_sha256']['validation']

SUBSET_SIZES = {'50pct': 5295, '25pct': 2648, '10pct': 1059, '5pct': 530}
TRAIN_SUBSETS = {}
subset_content_hashes = {}
for name, size in SUBSET_SIZES.items():
    path = SELECTION_DIR / f'subset_{name}.json'
    assert sha256(path) == provenance['selection_manifest_sha256'][name]
    manifest = json.loads(path.read_text())
    assert manifest['method'] == 'stratified_random' and manifest['selection_seed'] == 42
    assert manifest['dataset_revision'] == DATASET_REVISION and manifest['source_split'] == 'train'
    indices = manifest['row_indices']
    assert len(indices) == len(set(indices)) == size
    assert all(type(i) is int and 0 <= i < len(full_train) for i in indices)
    assert indices == sorted(indices)
    selected = full_train.select(indices)
    assert list(selected['id']) == manifest['ids'] == train_frame.iloc[indices]['id'].tolist()
    subset_content_hashes[name] = content_hash(selected)
    TRAIN_SUBSETS[name] = selected

def make_messages(claim, evidence):
    return [
        {'role': 'system', 'content': input_config['system_prompt']},
        {'role': 'user', 'content': f'Claim: {claim}\n\nEvidence: {evidence}'},
    ]

label_lookup = {label.casefold(): label for label in LABELS}

def parse_prediction(response):
    return label_lookup.get(' '.join(response.strip().split()).casefold(), 'INVALID')

def training_collator(rows):
    padding = {'input_ids': tokenizer.pad_token_id, 'attention_mask': 0, 'labels': -100}
    batch = {
        key: pad_sequence([torch.tensor(row[key], dtype=torch.long) for row in rows],
                          batch_first=True, padding_value=value)
        for key, value in padding.items()
    }
    assert batch['input_ids'].shape[1] <= SEQUENCE_LIMIT
    return batch

def evaluate_labels(eval_model, frame, folder, split, fingerprint):
    folder.mkdir(parents=True, exist_ok=True)
    previous_padding, previous_mode = tokenizer.padding_side, eval_model.training
    records = []
    started = time.perf_counter()
    max_prompt_tokens = 0
    try:
        tokenizer.padding_side = 'left'
        eval_model.eval()
        with torch.inference_mode():
            for start in tqdm(range(0, len(frame), 8), desc=f'{folder.parent.name}: {split}', leave=False):
                rows = frame.iloc[start:start + 8]
                prompts = [tokenizer.apply_chat_template(
                    make_messages(row.claim, row.evidence), tokenize=False, add_generation_prompt=True,
                ) for row in rows.itertuples(index=False)]
                inputs = tokenizer(prompts, padding=True, truncation=False,
                                   add_special_tokens=False, return_tensors='pt').to(DEVICE)
                width = inputs['input_ids'].shape[1]
                max_prompt_tokens = max(max_prompt_tokens, width)
                limit = SEQUENCE_LIMIT if split == 'validation' else eval_model.config.max_position_embeddings
                assert width + generation_config.max_new_tokens <= limit
                generated = eval_model.generate(**inputs, generation_config=generation_config)
                responses = tokenizer.batch_decode(generated[:, width:], skip_special_tokens=True)
                for uid, gold, response in zip(rows['id'], rows['label'], responses):
                    records.append({'id': uid, 'true_label': gold,
                                    'predicted_label': parse_prediction(response), 'raw_response': response})
                del inputs, generated
        assert [r['id'] for r in records] == frame['id'].tolist()
        gold, predicted = [r['true_label'] for r in records], [r['predicted_label'] for r in records]
        metrics = {
            'evaluation_split': split, 'n_examples': len(records), 'run_config_sha256': fingerprint,
            'source_csv_sha256': dataset_metadata['files'][split]['sha256'],
            'accuracy': float(accuracy_score(gold, predicted)),
            'macro_f1': float(f1_score(gold, predicted, labels=LABELS, average='macro', zero_division=0)),
            'weighted_f1': float(f1_score(gold, predicted, labels=LABELS, average='weighted', zero_division=0)),
            'invalid_rate': sum(p == 'INVALID' for p in predicted) / len(gold),
            'max_prompt_tokens': max_prompt_tokens, 'truncation': False,
            'elapsed_seconds': time.perf_counter() - started,
        }
        with (folder / 'predictions.jsonl').open('w', encoding='utf-8') as handle:
            for record in records:
                handle.write(json.dumps(record) + '\n')
        save_json(folder / 'classification_report.json', classification_report(
            gold, predicted, labels=LABELS, output_dict=True, zero_division=0))
        matrix_labels = LABELS + ['INVALID']
        pd.DataFrame(confusion_matrix(gold, predicted, labels=matrix_labels),
                     index=matrix_labels, columns=matrix_labels).loc[LABELS].to_csv(folder / 'confusion_matrix.csv')
        save_json(folder / 'metrics.json', metrics)
        return metrics
    finally:
        tokenizer.padding_side = previous_padding
        eval_model.train(previous_mode)
        gc.collect()
        torch.cuda.empty_cache()

class SingleGPUArguments(TrainingArguments):
    @property
    def n_gpu(self):
        return 1

class RandomSubsetTrainer(Trainer):
    def evaluate(self, eval_dataset=None, ignore_keys=None, metric_key_prefix='eval', **kwargs):
        result = evaluate_labels(self.model, validation_frame,
                                 Path(self.args.output_dir) / 'validation' / f'step_{self.state.global_step}',
                                 'validation', self.run_fingerprint)
        metrics = {f'{metric_key_prefix}_{key}': result[key]
                   for key in ['accuracy', 'macro_f1', 'weighted_f1', 'invalid_rate']}
        metrics[f'{metric_key_prefix}_runtime'] = result['elapsed_seconds']
        self.log(metrics)
        self.control = self.callback_handler.on_evaluate(self.args, self.state, self.control, metrics)
        return metrics

def run_subset(name):
    run_dir = OUTPUT_DIR / f'TinyLlama_Random_{name}_seed42'
    run_dir.mkdir(parents=True, exist_ok=True)
    train_data = TRAIN_SUBSETS[name]
    steps_per_epoch = math.ceil(math.ceil(len(train_data) / MICRO_BATCH) / GRAD_ACCUM)
    expected_steps = EPOCHS * steps_per_epoch
    assert expected_steps == {'50pct': 498, '25pct': 249, '10pct': 102, '5pct': 51}[name]
    run_config = {
        **baseline_config,
        'method': 'stratified_random', 'subset': name, 'selection_seed': 42,
        'training_rows': len(train_data),
        'selection_sha256': sha256(SELECTION_DIR / f'subset_{name}.json'),
        'training_content_sha256': subset_content_hashes[name],
        'validation_content_sha256': provenance['verified_content_sha256']['validation'],
        'source_training_fingerprint': preprocessing['splits']['train']['dataset_fingerprint'],
        'initial_adapter_files_sha256': initial_hashes,
        'warmup_steps': math.ceil(0.03 * expected_steps), 'expected_optimizer_steps': expected_steps,
        'generation_config': generation_config.to_dict(), 'package_versions': package_versions,
        'source_full_run_config_sha256': json_hash(baseline_config),
    }
    # The full-data fingerprint is not the identity of a selected subset.
    run_config.pop('training_fingerprint', None)
    config_path = run_dir / 'run_config.json'
    if config_path.exists():
        assert json.loads(config_path.read_text()) == run_config, f'Configuration differs: {run_dir}'
    else:
        save_json(config_path, run_config)
    fingerprint = json_hash(run_config)
    completion_path = run_dir / 'training_complete.json'
    test_path = run_dir / 'test/metrics.json'
    complete = json.loads(completion_path.read_text()) if completion_path.exists() else None
    if complete is not None:
        assert complete['run_config_sha256'] == fingerprint
        assert sha256(run_dir / 'best_adapter/adapter_model.safetensors') == complete['best_adapter_sha256']
    if test_path.exists():
        assert complete is not None, 'Test metrics exist without a training completion record.'
        metrics = json.loads(test_path.read_text())
        assert metrics['run_config_sha256'] == fingerprint
        assert metrics['source_csv_sha256'] == dataset_metadata['files']['test']['sha256']
        assert metrics['n_examples'] == 1823
        for filename in ['predictions.jsonl', 'classification_report.json', 'confusion_matrix.csv']:
            assert (run_dir / 'test' / filename).is_file(), filename
        print(f'{name}: reusing completed training and test results.')
        return complete, metrics

    local_base = local_model = local_trainer = None
    try:
        gc.collect()
        torch.cuda.empty_cache()
        set_seed(SEED)
        tokenizer.padding_side = 'right'
        print(f'\nTinyLlama random {name}: {len(train_data):,} training rows')
        local_base = AutoModelForCausalLM.from_pretrained(
            MODEL_ID, revision=MODEL_REVISION, dtype=torch.float16,
            attn_implementation='sdpa', device_map={'': 'cuda:0'},
        )
        adapter_path = run_dir / 'best_adapter' if complete is not None else INITIAL_ADAPTER
        local_model = PeftModel.from_pretrained(local_base, str(adapter_path), is_trainable=complete is None)
        local_model.config.use_cache = False
        local_model.config.pad_token_id = tokenizer.pad_token_id
        local_model.generation_config = generation_config
        local_model.gradient_checkpointing_disable()
        if complete is None:
            for parameter_name, parameter in local_model.named_parameters():
                if parameter.requires_grad:
                    assert 'lora_' in parameter_name
                    parameter.data = parameter.data.float()
            del parameter
            assert sum(p.numel() for p in local_model.parameters() if p.requires_grad) == 12615680
            args = SingleGPUArguments(
                output_dir=str(run_dir), num_train_epochs=EPOCHS,
                per_device_train_batch_size=MICRO_BATCH, gradient_accumulation_steps=GRAD_ACCUM,
                learning_rate=baseline_config['learning_rate'], warmup_steps=run_config['warmup_steps'],
                lr_scheduler_type='linear', optim='adamw_torch', weight_decay=0.01, max_grad_norm=1.0,
                fp16=True, bf16=False, gradient_checkpointing=False,
                eval_strategy='epoch', save_strategy='epoch', save_total_limit=2, save_only_model=False,
                load_best_model_at_end=True, metric_for_best_model='macro_f1', greater_is_better=True,
                logging_steps=25, logging_first_step=True, report_to='none',
                dataloader_num_workers=0, dataloader_pin_memory=True,
                seed=SEED, data_seed=SEED, label_names=['labels'],
            )
            assert args.world_size == 1 and args.device == DEVICE and args.train_batch_size == MICRO_BATCH
            (run_dir / 'training_args.json').write_text(args.to_json_string(), encoding='utf-8')
            local_trainer = RandomSubsetTrainer(
                model=local_model, args=args, train_dataset=train_data, eval_dataset=full_validation,
                data_collator=training_collator, processing_class=tokenizer,
            )
            local_trainer.run_fingerprint = fingerprint
            checkpoint = get_last_checkpoint(str(run_dir))
            if checkpoint is not None:
                for filename in ['adapter_model.safetensors', 'adapter_config.json',
                                 'optimizer.pt', 'scheduler.pt', 'trainer_state.json',
                                 'rng_state.pth', 'scaler.pt']:
                    assert (Path(checkpoint) / filename).is_file(), (
                        f'Incomplete resume checkpoint: {checkpoint}/{filename}'
                    )
            print('Starting from:', checkpoint or INITIAL_ADAPTER)
            print('Expected optimizer steps:', expected_steps)
            torch.cuda.reset_peak_memory_stats(DEVICE)
            result = local_trainer.train(resume_from_checkpoint=checkpoint)
            assert local_trainer.state.global_step == expected_steps
            assert local_trainer.state.epoch >= EPOCHS - 1e-6
            assert local_trainer.state.best_metric is not None
            validation_scores = [entry['eval_macro_f1']
                                 for entry in local_trainer.state.log_history
                                 if 'eval_macro_f1' in entry]
            assert len(validation_scores) == EPOCHS
            assert abs(max(validation_scores) - local_trainer.state.best_metric) < 1e-12
            local_trainer.save_model(str(run_dir / 'best_adapter'))
            tokenizer.save_pretrained(str(run_dir / 'best_adapter'))
            local_trainer.save_metrics('train', result.metrics)
            local_trainer.save_state()
            complete = {
                'run_config_sha256': fingerprint,
                'best_validation_macro_f1': float(local_trainer.state.best_metric),
                'best_checkpoint': local_trainer.state.best_model_checkpoint,
                'global_step': int(local_trainer.state.global_step), 'epoch': float(local_trainer.state.epoch),
                'train_call_seconds': result.metrics['train_runtime'],
                'peak_allocated_gib': torch.cuda.max_memory_allocated(DEVICE) / 1024**3,
                'best_adapter_sha256': sha256(run_dir / 'best_adapter/adapter_model.safetensors'),
                'resumed_from': checkpoint, 'runtime_note': 'A resumed call excludes earlier sessions.',
            }
            save_json(completion_path, complete)
            local_model = local_trainer.model
        # Test data is read only after the validation-selected adapter is fixed.
        test_frame = load_original('test', 'healthver_test.csv')
        assert len(test_frame) == 1823
        assert set(test_frame['id']).isdisjoint(train_frame['id'])
        assert set(test_frame['id']).isdisjoint(validation_frame['id'])
        metrics = evaluate_labels(local_model, test_frame, run_dir / 'test', 'test', fingerprint)
        print(f"Best validation F1: {complete['best_validation_macro_f1']:.4f} | "
              f"Test F1: {metrics['macro_f1']:.4f} | Test accuracy: {metrics['accuracy']:.4f}")
        return complete, metrics
    finally:
        local_trainer = local_model = local_base = None
        gc.collect()
        torch.cuda.empty_cache()

summary_rows = []
for subset_name in ['50pct', '25pct', '10pct', '5pct']:
    completed, metrics = run_subset(subset_name)
    summary_rows.append({
        'model': MODEL_ID, 'method': 'stratified_random', 'subset': subset_name,
        'training_rows': SUBSET_SIZES[subset_name], 'selection_seed': 42, 'training_seed': SEED,
        'best_validation_macro_f1': completed['best_validation_macro_f1'],
        'test_accuracy': metrics['accuracy'], 'test_macro_f1': metrics['macro_f1'],
        'test_weighted_f1': metrics['weighted_f1'], 'test_invalid_rate': metrics['invalid_rate'],
        'train_call_seconds': completed['train_call_seconds'], 'test_seconds': metrics['elapsed_seconds'],
    })
    summary = pd.DataFrame(summary_rows)
    summary_path = OUTPUT_DIR / 'random_results.csv'
    temporary = summary_path.with_suffix('.tmp.csv')
    summary.to_csv(temporary, index=False)
    temporary.replace(summary_path)
    print(summary.to_string(index=False))

print('\nAll four runs complete.')
print('Results:', OUTPUT_DIR / 'random_results.csv')
print('Selections and training settings remained fixed throughout the loop.')



TinyLlama random 50pct: 5,295 training rows


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

... (27 outputs left out)

validation: validation:   0%|          | 0/240 [00:00<?, ?it/s]

validation: validation:   0%|          | 0/240 [00:00<?, ?it/s]

TinyLlama_Random_5pct_seed42: test:   0%|          | 0/228 [00:00<?, ?it/s]

Best validation F1: 0.4158 | Test F1: 0.4432 | Test accuracy: 0.4668
                             model            method subset  training_rows  selection_seed  training_seed  best_validation_macro_f1  test_accuracy  test_macro_f1  test_weighted_f1  test_invalid_rate  train_call_seconds  test_seconds
TinyLlama/TinyLlama-1.1B-Chat-v1.0 stratified_random  50pct           5295              42             42                  0.764978       0.765222       0.755104          0.764225                0.0           2055.1644    119.949633
TinyLlama/TinyLlama-1.1B-Chat-v1.0 stratified_random  25pct           2648              42             42                  0.661297       0.682940       0.656447          0.677086                0.0           1230.7020    120.475280
TinyLlama/TinyLlama-1.1B-Chat-v1.0 stratified_random  10pct           1059              42             42                  0.547351       0.579265       0.532562          0.566237                0.0            721.8365    120.113811